In [2]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(x,k):
    h,w=k.shape[0],k.shape[1]
    Y=torch.zeros((x.shape[0]-h+1,x.shape[1]-w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j]=(x[i:i+h,j:j+w]*k).sum()
    return Y

In [3]:
X=torch.tensor([[0.0,1.0,2.0],
                [3.0,4.0,5.0],
                [6.0,7.0,8.0]])
k=torch.tensor([[0.0,1.0],[2.0,3.0]])
corr2d(X,k)

tensor([[19., 25.],
        [37., 43.]])

In [4]:
class Conv2d(nn.Module):
    def __init__(self,kernel_size):
        super.__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = torch.Parameter(torch.zeros(1))

    def forword(self,x):
        return corr2d(x,self.weight)+self.bias

In [5]:
x = torch.ones((6,8))
x[:,2:6] = 0.0
x

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [6]:
k = torch.tensor([[1.0,-1.0]])
Y=corr2d(x,k)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [7]:
conv2d = nn.Conv2d(1,1,kernel_size=(1,2),bias=False)

x=x.reshape((1,1,6,8))
Y=Y.reshape((1,1,6,7))

for i in range(10):
    Y_hat =  conv2d(x)
    l=(Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:]-=3e-2 * conv2d.weight.grad
    if (i+1)%2==0:
        print(f'batch: {i+1},loss: {l.sum():.3f}')

batch: 2,loss: 12.937
batch: 4,loss: 3.539
batch: 6,loss: 1.154
batch: 8,loss: 0.423
batch: 10,loss: 0.165


In [8]:
conv2d.weight.data.reshape((1,2))

tensor([[ 0.9448, -1.0272]])

In [9]:
def comp_conv2d(conv2d,X):
    X=X.reshape((1,1)+X.shape)
    Y=conv2d(X)
    return Y.reshape(Y.shape[2:])

conv2d = nn.Conv2d(1,1,kernel_size=3,padding=1)
X=torch.rand(size=(8,8))
comp_conv2d(conv2d,X).shape

torch.Size([8, 8])

In [10]:
conv2d = nn.Conv2d(1,1,kernel_size=(5,3),padding=(2,1),stride = 2)
comp_conv2d(conv2d,X).shape

torch.Size([4, 4])

In [11]:
conv2d = nn.Conv2d(1,1,kernel_size=(5,3),padding=(0,1),stride = (2,5))
comp_conv2d(conv2d,X).shape

torch.Size([2, 2])

In [17]:
def corr2d_muti_in(X,K):
    return sum(d2l.corr2d(x,k)for x,k in zip(X,K))
X=torch.tensor([[[0.0,1.0,2.0],[3.0,4.0,5.0],[6.0,7.0,8.0]],
                [[1.0,2.0,3.0],[4.0,5.0,6.0],[7.0,8.0,9.0]]])    
K=torch.tensor([[[0.0,1.0],[2.0,3.0]],
                [[4.0,5.0],[6.0,7.0]]])
corr2d_muti_in(X,K)

tensor([[ 92., 120.],
        [176., 204.]])

In [18]:
def corr2d_multi_in_out(X,K):
    return torch.stack([corr2d_muti_in(X,k)for k in K],0)

K=torch.stack((K,K+1,K+2),0)
corr2d_muti_in_out(X,K)

tensor([[[ 92., 120.],
         [176., 204.]],

        [[112., 148.],
         [220., 256.]],

        [[132., 176.],
         [264., 308.]]])

In [19]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

In [20]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

In [21]:
def pool2d(X,pool_size,mode='max'):
    p_h,p_w=pool_size
    Y = torch.zeros((X.shape[0]-p_h+1,X.shape[1]-p_w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode =='max':
                Y[i,j] = X[i:i + p_h,j:j+p_w].max()
            elif mode =='avg':
                Y[i,j] = X[i:i + p_h, j:j + p_w].mean()
    return Y            

In [23]:
X = torch.tensor([[0.0,1.0,2.0],[3.0,4.0,5.0],[6.0,7.0,8.0]])
pool2d(X,(2,2))

tensor([[4., 5.],
        [7., 8.]])

In [24]:
pool2d(X,(2,2),'avg')

tensor([[2., 3.],
        [5., 6.]])

In [25]:
X = torch.arange(16,dtype = torch.float32).reshape((1,1,4,4))
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])

In [27]:
pool2d = nn.MaxPool2d(3)
pool2d(X)

tensor([[[[10.]]]])

In [28]:
pool2d = nn.MaxPool2d(3,padding = 1,stride = 2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])

In [29]:
X = torch.cat((X,X+1),1)
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]],

         [[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])

In [30]:
pool2d = nn.MaxPool2d(3,padding = 1,stride = 2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])